In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns
import numpy as np

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-30B-A3B", trust_remote_code=True)

In [ ]:
score_files = glob.glob('data/lmsys/routing_scores/*.parquet')
dfs = pd.concat([pd.read_parquet(f) for f in score_files], ignore_index=True)

In [ ]:
dft = pd.read_parquet('data/lmsys/request_text/all_request_text_data.parquet')
dft['token_str'] = tokenizer.batch_decode(dft['token_actual_id'], clean_up_tokenization_spaces=True)

In [ ]:
num_layers = dfs['layer_id'].unique().size
num_layers

In [ ]:
num_experts = dfs['expert_id'].unique().size
num_experts

In [ ]:
dft.columns

In [ ]:
dfs.columns

In [ ]:
tids = tokenizer("hello").input_ids
assert len(tids) == 1
tid = tids[0]

prim_keys = ['request_id', 'token_position_in_sequence']

matched = dft[dft['token_actual_id'] == tid][prim_keys]

matched

In [ ]:
token_scores = matched.merge(dfs, on=prim_keys, how='left')

token_scores

In [ ]:
req_to_plot = 210
token_to_plot = 41

# df_filtered = token_scores[(token_scores['request_id'] == req_to_plot) & (token_scores['token_position_in_sequence'] == token_to_plot)]

df_filtered = dfs[(dfs['request_id'] == req_to_plot) & (dfs['token_position_in_sequence'] == token_to_plot)]

if df_filtered.empty:
    print(f"错误：在DataFrame中找不到 token_position_in_sequence = {token_to_plot} 的数据。")
else:
    # 3. 创建数据透视表 (pivot table)
    # 横向是 expert_id，纵向是 layer_id，值为 routing_score
    # 我们需要处理可能存在的重复项 (layer_id, expert_id 对)，这里我们取平均值
    heatmap_data = pd.pivot_table(
        df_filtered,
        values='routing_score',
        index='layer_id',
        columns='expert_id',
        aggfunc='mean' # 如果每个(layer_id, expert_id)只有一个routing_score，用 'first' 或 'mean' 都可以
    )

    if heatmap_data.empty:
        print(f"错误：无法为 token_position_in_sequence = {token_to_plot} 创建热力图数据。检查筛选条件和数据。")
    else:
        # 4. 使用 seaborn 绘制热力图 (seaborn 基于 matplotlib，更易用)
        plt.figure(figsize=(100, 60)) #调整图像大小
        ax = sns.heatmap(heatmap_data, annot=True, fmt=".4f", cmap="viridis", linewidths=.5)

        # 5. 设置图表标题和坐标轴标签
        plt.title(f'Routing Score Heatmap for Req {req_to_plot} Token {token_to_plot}')
        plt.xlabel('Expert ID')
        plt.ylabel('Layer ID')

        # 6. 高亮 Top-K 专家
        def mark_pos(target_layer, target_expert):
            if target_layer in heatmap_data.index and target_expert in heatmap_data.columns:
                row_idx_highlight = heatmap_data.index.get_loc(target_layer)
                col_idx_highlight = heatmap_data.columns.get_loc(target_expert)
                ax.add_patch(plt.Rectangle((col_idx_highlight, row_idx_highlight), 1, 1,
                                        fill=False, edgecolor='red', lw=3))
        
        scores_data = heatmap_data.to_numpy()
        top_k = 8
        experts_for_layer = scores_data.argpartition(-top_k, axis=1)[:, -top_k:]
        for layer in heatmap_data.index:
            for expert in range(top_k):
                expert_id = experts_for_layer[layer][expert].item()
                mark_pos(layer, expert_id)

        # 7. 显示图表
        plt.tight_layout() # 自动调整布局，防止标签重叠
        # plt.savefig('data/images/substrings-substring.png')
        plt.show()

In [ ]:
print(tokenizer.decode(dft[dft['request_id'] == 1]['token_actual_id']))

In [ ]:
dft[dft['request_id'] == 0]

In [ ]:
tokenizer.encode('</think>')

In [ ]:
tokenizer.decode(14990)

In [ ]:
tokenizer.decode(23811)

In [ ]:
req_to_plot = 210
layer_to_plot = 11

df_filtered = dfs[dfs['request_id'] == req_to_plot]
df_filtered = df_filtered[df_filtered['layer_id'] == layer_to_plot]
df_filtered = df_filtered[df_filtered['token_position_in_sequence'] < 200]

df_tokens = dft[dft['request_id'] == req_to_plot]
df_tokens = df_tokens[df_tokens['token_position_in_sequence'] < 200]
df_tokens = df_tokens.sort_values(by='token_position_in_sequence')
df_tokens = df_tokens['token_str'].to_numpy()

if df_filtered.empty:
    print(f"错误：在DataFrame中找不到 layer_id = {layer_to_plot} 的数据。")
else:
    # 3. 创建数据透视表 (pivot table)
    # 横向是 expert_id，纵向是 token_position_in_sequence，值为 routing_score
    # 我们需要处理可能存在的重复项 (token_position_in_sequence, expert_id 对)，这里我们取平均值
    heatmap_data = pd.pivot_table(
        df_filtered,
        values='routing_score',
        index='token_position_in_sequence',
        columns='expert_id',
        aggfunc='mean'
    )

    if heatmap_data.empty:
        print(f"错误：无法为 layer_id = {layer_to_plot} 创建热力图数据。检查筛选条件和数据。")
    else:
        # 4. 使用 seaborn 绘制热力图 (seaborn 基于 matplotlib，更易用)
        plt.figure(figsize=(100, 60)) #调整图像大小
        ax = sns.heatmap(
            heatmap_data, 
            annot=True, 
            fmt=".4f", 
            cmap="viridis", 
            linewidths=.5,
            norm=LogNorm()
        )

        # 5. 设置图表标题和坐标轴标签
        plt.title(f'Routing Score Heatmap for Req {req_to_plot} Layer {layer_to_plot}')
        plt.xlabel('Expert ID')
        plt.ylabel('Token ID')

        ax.set_yticklabels(
            [f'{i} {s}' for i, s in zip(heatmap_data.index, df_tokens)],
            rotation=0,  # 设置标签旋转角度为0
            fontsize=12,  # 设置字体大小
        )

        # 6. 高亮 Top-K 专家
        def mark_pos(target_row, target_expert):
            if target_row in heatmap_data.index and target_expert in heatmap_data.columns:
                row_idx_highlight = heatmap_data.index.get_loc(target_row)
                col_idx_highlight = heatmap_data.columns.get_loc(target_expert)
                ax.add_patch(plt.Rectangle((col_idx_highlight, row_idx_highlight), 1, 1,
                                        fill=False, edgecolor='red', lw=3))
        
        scores_data = heatmap_data.to_numpy()
        top_k = 4
        experts_for_row = scores_data.argpartition(-top_k, axis=1)[:, -top_k:]
        for row in heatmap_data.index:
            for expert in range(top_k):
                expert_id = experts_for_row[row][expert].item()
                mark_pos(row, expert_id)

        # 7. 显示图表
        plt.tight_layout() # 自动调整布局，防止标签重叠
        # plt.savefig(f'data/lmsys/images/affine/req{req_to_plot}_layer{layer_to_plot}_top{top_k}.png')
        plt.show()

In [ ]:
# Stat total expert activation count for reqs
req_to_plot = 210
df = dfs[dfs['request_id'] == req_to_plot]
scores = pd.pivot_table(
    df,
    values='routing_score',
    index=['layer_id', 'token_position_in_sequence'],
    columns='expert_id',
    aggfunc='mean'
)
# [layers, tokens, experts]
scores = scores.to_numpy().reshape(num_layers, -1, num_experts)

# Select Top-k
top_k = 4
top_k_experts = np.argsort(scores, axis=2)[:, :, -top_k:]

# Count activations
activation_counts = np.zeros((num_layers, num_experts), dtype=int)
np.add.at(activation_counts, (np.arange(num_layers)[:, None, None], top_k_experts), 1)
activation_counts

# Heatmap of activation counts
plt.figure(figsize=(60, 30))
sns.heatmap(activation_counts, annot=True, fmt="d", cmap="viridis", linewidths=.5)
plt.title(f'Expert Activation Counts for Request {req_to_plot} SeqLen {scores.shape[1]}')
plt.xlabel('Expert ID')
plt.ylabel('Layer ID')
plt.tight_layout()
# plt.savefig(f'data/lmsys/images/activation_counts_req{req_to_plot}.png')
plt.show()


In [ ]:
activation_counts.max(axis=1) / scores.shape[1]